### Day 16 · 数据摄取(CSV/JSON/SQL)


**pandas** 提供 `read_csv`、`read_json`、`read_sql` 从多种数据源读入数据,以及 `to_csv`、`to_json`、`to_sql` 将处理后的数据导出。每个 cell 独立 import、独立运行、数据硬编码。


#### read_csv 全部参数

[痛点] 拿到一份 CSV 文件直接 `pd.read_csv("data.csv")` 经常报错: 中文乱码、多余的表头行、缺失标记未被识别。原因就是没有指定关键参数。

[类比] 读 CSV 像点外卖 -- `encoding` 是语言(中/英文菜单)、`sep` 是分隔方式(筷子/叉子)、`usecols` 是只点想吃的菜、`nrows` 是先尝一口。

[解释] `read_csv` 最核心的 6 个参数: `encoding` 指定字符编码(中文常用 utf-8 或 gbk)、`sep` 指定列分隔符(默认逗号)、`index_col` 指定某列作为行索引、`usecols` 只读取指定列(省内存)、`nrows` 只读前 N 行(预览大文件)、`na_values` 把特定字符串(如 "NA"、"-")也视为缺失值。


In [ ]:
import pandas as pd

# 构造示例数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [20, 21, 19, 22],
    "成绩": [88.5, 92.0, 76.5, 81.0]
})

# 写出 CSV 文件,模拟真实数据源
df.to_csv("students.csv", index=False, encoding="utf-8")

# --- 执行过程 ---
# ① to_csv: 将 DataFrame 写出为磁盘文件,index=False 不导出行号
df_read = pd.read_csv("students.csv", encoding="utf-8")
# ② read_csv: 按 utf-8 编码读取文件,自动识别第 0 行为列名
# ③ print: 打印读取结果,验证与原始 DataFrame 一致
print("--- 基本读取 ---")
print(df_read)


In [ ]:
import pandas as pd

# 写出以分号分隔的 CSV 文件
with open("semi.csv", "w", encoding="utf-8") as f:
    f.write("姓名;年龄;成绩\n")
    f.write("张三;20;88.5\n")
    f.write("李四;21;92.0\n")
    f.write("王五;19;76.5\n")

# --- 执行过程 ---
# ① open: 以写入模式创建文件,encoding 确保中文正常写入
# ② read_csv: sep=";" 指定分隔符,index_col=0 把姓名列设为行索引
# ③ usecols: 只加载"姓名"和"成绩"两列,跳过"年龄"列
# ④ nrows=2: 只读取前 2 行,适合预览大文件
df = pd.read_csv("semi.csv",
                 sep=";",
                 index_col=0,
                 usecols=["姓名", "成绩"],
                 nrows=2,
                 encoding="utf-8")
print("--- sep / index_col / usecols / nrows ---")
print(df)


**逐行解剖**

- `sep=";"`: 告诉 pandas 列与列之间用分号分隔,默认是逗号。读取导出软件生成的 CSV 时常见分号或制表符。
- `index_col=0`: 把第 0 列(姓名)设为行索引,这样查某人成绩时可以用 `df.loc["张三"]`。
- `usecols=["姓名", "成绩"]`: 只加载需要的列,数据量大时能显著节省内存。
- `nrows=2`: 只读取前 2 行,适合预览几百 MB 的大文件。


In [ ]:
import pandas as pd

# 构造含特殊缺失标记的数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [20, "NA", "-", 22]
})
df.to_csv("na_demo.csv", index=False, encoding="utf-8")

# --- 执行过程 ---
# ① read_csv 不指定 na_values: 所有值均按原样读入,"NA" 和 "-" 被当字符串处理
df_default = pd.read_csv("na_demo.csv", encoding="utf-8")
print("--- 不指定 na_values ---")
print(df_default)
print("年龄列类型:", df_default["年龄"].dtype)

# ② read_csv 指定 na_values: 将 "NA" 和 "-" 识别为缺失值 NaN
df_na = pd.read_csv("na_demo.csv",
                    na_values=["NA", "-"],
                    encoding="utf-8")
print("")
print("--- na_values=[NA,\'-\'] ---")
print(df_na)
print("年龄列类型:", df_na["年龄"].dtype)


**逐行解剖**

- `na_values=["NA", "-"]`: 列出所有应被视为缺失值的字符串。默认 pandas 只识别空字符串、NaN、NA、NULL 等少数标记,实际数据中常见 "?"、"-"、"N/A" 都需要手动指定。
- 指定后,这些位置会变成 NaN,列类型也会正确变为 float64,否则会被当作字符串。


**练习 1**

某班级成绩 CSV(`score.csv`)使用制表符(`\\t`)分隔,第 3 行才是列名,且当成绩为 "缺考" 时应视为缺失值。请用 `pd.read_csv` 正确读入,并只读取 "姓名" 和 "数学" 两列,最多读 10 行。

> 问自己:
> - 制表符对应哪个参数?
> - "缺考" 应该传给哪个参数?


In [ ]:
import pandas as pd

# 模拟写出该 CSV 文件
with open("score.csv", "w", encoding="utf-8") as f:
    f.write("这是一份班级成绩表\n")
    f.write("日期:2026-07-08\n")
    f.write("姓名\t语文\t数学\t英语\n")
    f.write("张三\t80\t90\t85\n")
    f.write("李四\t缺考\t78\t92\n")
    f.write("王五\t88\t缺考\t76\n")

# ======================
# 学员代码区
# ======================
pass

# ======================
# 参考答案
# ======================
df = pd.read_csv("score.csv",
                 sep="\\t",
                 header=2,
                 usecols=["姓名", "数学"],
                 nrows=10,
                 na_values=["缺考"],
                 encoding="utf-8")
print(df)


#### read_json orient 与嵌套

[痛点] 从 API 拿到的 JSON 数据经常是多层嵌套结构,直接 `pd.read_json()` 报错 -- 因为 pandas 默认只识别几种固定格式。

[类比] JSON 像一封信: 平铺的列表套字典就是"白话信"(直接读),嵌套的字典套字典就是"加密信"(需要解密/展平)。

[解释] `orient` 参数告诉 pandas JSON 数据的排列方式。最常用 `orient="records"`,即 `[{"列名":值}, ...]` 格式。对于嵌套 JSON,需要用 `json_normalize` 展平内部字段。


In [ ]:
import pandas as pd

# 构造示例数据
df = pd.DataFrame({
    "姓名": ["张三", "李四"],
    "年龄": [20, 21]
})

# --- 执行过程 ---
# ① to_json(orient="records"): 将 DataFrame 转成 JSON 字符串,
#    每条记录变成 {"姓名":"...", "年龄":...},整体是列表
json_str = df.to_json(orient="records", force_ascii=False)
print("--- JSON 字符串 ---")
print(json_str)

# ② read_json: 从 JSON 字符串读回 DataFrame,自动识别 orient
df_back = pd.read_json(json_str)
print("")
print("--- read_json 读回 ---")
print(df_back)


In [ ]:
import pandas as pd

# 嵌套结构: 每个学生的"成绩"字段是一个子字典
data = [
    {"姓名": "张三", "班级": "一班",
     "成绩": {"语文": 88, "数学": 92}},
    {"姓名": "李四", "班级": "二班",
     "成绩": {"语文": 75, "数学": 85}}
]

# --- 执行过程 ---
# ① json_normalize: 把嵌套字典展平,meta 指定保留的外层字段
df = pd.json_normalize(data, meta=["姓名", "班级"])
print("--- 展平后(保留 meta) ---")
print(df)

# ② 不指定 meta: 自动展平所有层级,列名用"."连接
df2 = pd.json_normalize(data)
print("")
print("--- 完全展平 ---")
print(df2)


**逐行解剖**

- `orient="records"`: 告诉 pandas,JSON 是一个列表,每个元素是一条记录(字典)。这是 Web API 最常见的格式。
- `force_ascii=False`: 默认 pandas 会把中文转成 Unicode 转义形式,设为 False 可保留原始中文。
- `pd.json_normalize(data, meta=["姓名","班级"])`: 把嵌套的字典"拍平"成多个列,`meta` 指定保留的外层字段。


**练习 2**

以下 JSON 字符串记录了 3 位员工的姓名和城市,请用 `pd.read_json` 将其转为 DataFrame,然后再把姓名转为行索引。

> 问自己:
> - `read_json` 能直接读 JSON 字符串吗?
> - 把某列设为行索引用什么方法?


In [ ]:
import pandas as pd

json_str = (
    '[{"姓名": "张三", "城市": "北京"}, '
    '{"姓名": "李四", "城市": "上海"}, '
    '{"姓名": "王五", "城市": "广州"}]'
)

# ======================
# 学员代码区
# ======================
pass

# ======================
# 参考答案
# ======================
df = pd.read_json(json_str)
df = df.set_index("姓名")
print(df)


#### read_sql 与 sqlite3

[痛点] 数据存在 SQLite 数据库中,不知道如何用 pandas 读取 -- 需要先连数据库、写 SQL、再转成 DataFrame。

[类比] 数据库就像一个文件柜: `sqlite3.connect()` 是打开柜子的钥匙、`cursor` 是取文件的双手、`pd.read_sql()` 是把取出的文件整理成表格。

[解释] 标准流程: 连接数据库 -> 执行 SQL 查询 -> `pd.read_sql()` 把结果读成 DataFrame -> **用完关闭连接**。推荐用 `with` 上下文管理器自动关闭,避免连接泄漏。


In [ ]:
import pandas as pd
import sqlite3

# --- 执行过程 ---
# ① sqlite3.connect(":memory:"): 创建内存数据库,退出即丢失,适合测试
# ② with 语句: 退出代码块时自动调用 conn.close(),无需手动关闭
with sqlite3.connect(":memory:") as conn:
    cur = conn.cursor()
    # ③ cursor.execute: 创建 students 表
    cur.execute(
        "CREATE TABLE students "
        "(id INTEGER PRIMARY KEY, name TEXT, age INTEGER)"
    )
    rows = [
        (1, "张三", 20),
        (2, "李四", 21),
        (3, "王五", 19),
        (4, "赵六", 22)
    ]
    # ④ executemany: 批量插入,? 是占位符防止 SQL 注入
    cur.executemany(
        "INSERT INTO students VALUES (?,?,?)",
        rows
    )
    # ⑤ commit: 提交事务,使插入操作持久化
    conn.commit()

    # ⑥ read_sql: 把 SELECT 结果直接转为 DataFrame
    df = pd.read_sql("SELECT * FROM students", conn)
    print("--- 全部数据 ---")
    print(df)

    # ⑦ 带 WHERE 条件的查询
    df_where = pd.read_sql(
        "SELECT name, age FROM students WHERE age > 20",
        conn
    )
    print("")
    print("--- 年龄 > 20 ---")
    print(df_where)


**逐行解剖**

- `sqlite3.connect(":memory:")`: 创建内存数据库(重启即丢,适合测试),传入文件路径则持久化到磁盘。
- `cur.executemany(sql, rows)`: 批量执行同一条 SQL,`?` 是占位符,防止 SQL 注入。
- `conn.commit()`: 提交事务,插入/删除操作必须调用才生效。
- `pd.read_sql(sql, conn)`: 把 SQL 查询结果直接转为 DataFrame,比手动遍历 cursor 方便得多。
- `with` 语句: 退出时自动 `conn.close()`,是资源管理的最佳实践。


**练习 3**

已有 SQLite 数据库 `company.db`,其中包含一张 `employees` 表,列有 `id`、`name`、`salary`、`dept`。请使用 `with` 上下文管理器连接数据库,查询"技术部"所有员工的姓名和薪资,结果转为 DataFrame。

> 问自己:
> - `with` 语句的好处是什么?
> - SQL 查询中字符串值用什么符号包裹?


In [ ]:
import pandas as pd
import sqlite3

# 先创建模拟数据库并插入数据
with sqlite3.connect("company.db") as conn:
    cur = conn.cursor()
    cur.execute(
        "CREATE TABLE IF NOT EXISTS employees "
        "(id INTEGER PRIMARY KEY, name TEXT, salary INTEGER, dept TEXT)"
    )
    rows = [
        (1, "张三", 15000, "技术"),
        (2, "李四", 12000, "市场"),
        (3, "王五", 18000, "技术"),
        (4, "赵六", 11000, "财务")
    ]
    cur.executemany(
        "INSERT OR REPLACE INTO employees VALUES (?,?,?,?)",
        rows
    )
    conn.commit()

# ======================
# 学员代码区
# ======================
pass

# ======================
# 参考答案
# ======================
with sqlite3.connect("company.db") as conn:
    df = pd.read_sql(
        "SELECT name, salary FROM employees WHERE dept = '技术'",
        conn
    )
    print(df)


#### to_csv / to_json / to_sql 导出数据

[痛点] 数据处理完了要保存 -- 导出 CSV 时经常多出一列行索引,导出 JSON 时中文变成乱码,追加数据到数据库时不小心覆盖了旧表。

[类比] 导出数据像寄快递: `index=False` 是不寄包装盒(行索引)、`force_ascii=False` 是保留中文地址不翻译、`if_exists="append"` 是"追加"而不是"替换"。

[解释] `to_csv` 最常用,必须加 `index=False` 避免多余索引列; `to_json` 用 `orient="records"` 记录格式,`force_ascii=False` 保留中文; `to_sql` 的 `if_exists` 参数控制表已存在时如何处理(fail/replace/append)。


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, 21, 19]
})

# --- 执行过程 ---
# ① 默认 to_csv: 会多出第 0 列 "Unnamed: 0",内容是行索引 0,1,2
df.to_csv("with_index.csv", encoding="utf-8")
print("--- 默认导出(含行索引) ---")
print(pd.read_csv("with_index.csv"))

# ② index=False: 去掉行索引,导出干净的表格
df.to_csv("no_index.csv", index=False, encoding="utf-8")
print("")
print("--- index=False ---")
print(pd.read_csv("no_index.csv"))


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四"],
    "年龄": [20, 21]
})

# --- 执行过程 ---
# ① force_ascii=True(默认): 中文被转义成 \uXXXX 形式
print("--- force_ascii=True (默认) ---")
print(df.to_json(orient="records"))

# ② force_ascii=False: 保留中文原样
print("")
print("--- force_ascii=False ---")
print(df.to_json(orient="records", force_ascii=False))

# ③ 导出到文件,indent=2 让 JSON 可读
df.to_json("output.json", orient="records", force_ascii=False, indent=2)
print("")
print("--- 已写出 output.json ---")


In [ ]:
import pandas as pd
import sqlite3

df = pd.DataFrame({
    "name": ["张三", "李四", "王五"],
    "age": [20, 21, 19]
})

# --- 执行过程 ---
# ① sqlite3.connect: 连接数据库文件(不存在则创建)
with sqlite3.connect("export.db") as conn:
    # ② if_exists="replace": 表已存在则删掉重建
    df.to_sql("students", conn, if_exists="replace", index=False)
    print("--- replace 后读回 ---")
    print(pd.read_sql("SELECT * FROM students", conn))

    # ③ if_exists="append": 在已有表末尾追加新行
    df_new = pd.DataFrame({"name": ["赵六"], "age": [22]})
    df_new.to_sql("students", conn, if_exists="append", index=False)
    print("")
    print("--- append 后读回 ---")
    print(pd.read_sql("SELECT * FROM students", conn))


**逐行解剖**

- `index=False`: CSV 最常用参数,不导出默认的行号列。加 `encoding="utf-8"` 避免中文乱码。
- `orient="records"`: JSON 导出最常用格式,生成 `[{...}, {...}]` 形式的列表套字典。
- `force_ascii=False`: 让中文字符原样输出,不转成 Unicode 转义序列。
- `if_exists`: 控制目标表已存在时的行为 -- `fail` 报错(默认)、`replace` 删旧表建新表、`append` 追加数据。


**练习 4**

已有以下 DataFrame,请完成三项任务: 1. 导出为 CSV(`result.csv`),不含行索引、utf-8 编码; 2. 导出为 JSON(`result.json`),记录格式、保留中文、缩进 2 格; 3. 追加到 SQLite 数据库 `my.db` 的 `person` 表中(表已存在)。

> 问自己:
> - `to_csv` 的 `index` 参数应该填什么?
> - `to_sql` 的 `if_exists` 如果想覆盖旧表应该填什么?


In [ ]:
import pandas as pd
import sqlite3

df = pd.DataFrame({"姓名": ["陈七"], "年龄": [25]})

# 先在数据库中建一张 person 表
with sqlite3.connect("my.db") as conn:
    cur = conn.cursor()
    cur.execute(
        "CREATE TABLE IF NOT EXISTS person "
        "(name TEXT, age INTEGER)"
    )
    conn.commit()

# ======================
# 学员代码区
# ======================
pass

# ======================
# 参考答案
# ======================
# 任务 1: 导出 CSV
df.to_csv("result.csv", index=False, encoding="utf-8")

# 任务 2: 导出 JSON
df.to_json("result.json", orient="records",
           force_ascii=False, indent=2)

# 任务 3: 追加到数据库
with sqlite3.connect("my.db") as conn:
    df.to_sql("person", conn, if_exists="append", index=False)
    print("追加后内容:")
    print(pd.read_sql("SELECT * FROM person", conn))


**今日小结**

| 概念 | 解决的痛点 |
|---|---|
| read_csv 参数 | 中文乱码、多余表头、缺失标记未识别 |
| read_json + orient | JSON 格式不匹配报错、嵌套无法直接读 |
| read_sql + sqlite3 | 数据库数据无法直接分析 |
| to_csv / to_json / to_sql | 导出多余索引、中文乱码、表覆盖 vs 追加 |

**更多练习**

- 当堂练: course/lesson16/in_class/practice01-06.py
- 课后作业: course/lesson16/homework/task01-03.py
